Quyidagi masala **“minimal nechta teoremani isbotlab, 1-teoremani (asosiy) eng tez isbotlash”** haqida.

---

## 1) Masalaning modeli (graf)

Har bir teorema `Ti` o‘zidan oldin isbotlanishi kerak bo‘lgan teoremalar to‘plamiga ega: `deps[i]` (Ci ta).

**Qoidasi:** `Ti` ni isbotlash uchun `deps[i]` ichidan **kamida yarmi** isbotlangan bo‘lishi kerak.
Demak kerak bo‘ladigan minimum son:

[
need(i)=\lceil C_i/2 \rceil = (C_i+1)/2
]

**Muhim cheklovlar:**

* Kirishdagi barcha `Ai,j` lar butun fayl bo‘yicha **takrorlanmaydi** ⇒ **har bir teorema maksimum 1 ta teoremaning “bog‘liqligi” (dependency) bo‘la oladi**.
* Sikl yo‘q.

Bu shuni anglatadiki, teoremalar bog‘lanishi **daraxt(lar) / o‘rmon** ko‘rinishida, va teoremalarning “pastki” teoremalariga (dependency) ketadigan yo‘llar **bir-birini kesib o‘tmaydi**. Ya’ni tanlagan isbotlar “ustma-ust” ishlatilmaydi.

---

## 2) Asosiy g‘oya (DP)

`cost[i]` = `Ti` ni isbotlash uchun minimal nechta teorema isbotlash kerak (o‘zini ham qo‘shib).

* Agar `Ci = 0` bo‘lsa:
  `cost[i] = 1` (faqat o‘zi)
* Aks holda:

  * `need = ceil(Ci/2)`
  * `deps[i]` ichidan **need ta** teoremani tanlaymiz, lekin **ularni isbotlash narxi eng kichik bo‘lganlarini** tanlash foydali.
  * Shuning uchun:

[
cost[i] = 1 + \sum_{need \text{ ta eng kichik}} cost[dep]
]

Bu yerda “1” — `Ti` ning o‘zi.

**Nega aynan eng kichiklarini tanlaymiz?**
Chunki ortiqcha teorema isbotlash foyda bermaydi (faqat vaqtni oshiradi). Shuning uchun minimal bo‘lishi uchun kerakli `need` ta dependency dan eng arzonga tushadiganlarini olamiz.

---

## 3) Isbotlash tartibi (order)

Agar `Ti` uchun tanlangan dependencylar ro‘yxati `chosen[i]` bo‘lsa, isbotlash tartibi oddiy:

1. Avval `chosen[i]` dagilarni (ular ham o‘z dependencylarini) isbotlaymiz
2. So‘ng `i` ni isbotlaymiz

Bu **postorder** (bolalar → ota) yurish.

---

## 4) Murakkablik

* Har bir bog‘lanish 1 marta ishlatiladi, umumiy dependencylar soni `≤ N-1`.
* Har teorema uchun bolalar costlarini sort qilish: amalda `O(E log E)` atrofida, `N=10000` uchun bemalol.

---

## 5) C++ yechim (tushunarli, iterativ DFS)

```cpp
#include <bits/stdc++.h>
using namespace std;

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int N;
    cin >> N;

    vector<vector<int>> deps(N + 1);
    for (int i = 1; i <= N; i++) {
        int x;
        while (cin >> x && x != 0) {
            deps[i].push_back(x);
        }
    }

    // cost[i] - minimal number of theorems to prove theorem i (including itself)
    vector<int> cost(N + 1, -1);

    // chosen[i] - which dependencies we actually choose to prove for theorem i
    vector<vector<int>> chosen(N + 1);

    // --- Compute cost[] bottom-up with iterative postorder from node 1 ---
    // Because graph is acyclic, we can do DFS stack (node, state)
    // state=0 -> go down, state=1 -> compute
    vector<pair<int,int>> st;
    st.push_back({1, 0});

    // We may need to compute costs for nodes reachable from 1 through deps (full subtree).
    while (!st.empty()) {
        auto [v, state] = st.back();
        st.pop_back();

        if (state == 0) {
            if (cost[v] != -1) continue; // already computed
            st.push_back({v, 1});
            for (int u : deps[v]) {
                if (cost[u] == -1) st.push_back({u, 0});
            }
        } else {
            // compute cost[v]
            int c = (int)deps[v].size();
            if (c == 0) {
                cost[v] = 1;
                chosen[v].clear();
                continue;
            }

            int need = (c + 1) / 2; // ceil(c/2)

            // collect (cost[dep], dep)
            vector<pair<int,int>> arr;
            arr.reserve(c);
            for (int u : deps[v]) {
                arr.push_back({cost[u], u});
            }
            nth_element(arr.begin(), arr.begin() + need, arr.end()); 
            // after nth_element, first need elements are the smallest (unordered)

            // take the smallest 'need' elements
            chosen[v].clear();
            int sum = 0;
            for (int i = 0; i < need; i++) {
                sum += arr[i].first;
                chosen[v].push_back(arr[i].second);
            }
            cost[v] = 1 + sum;
        }
    }

    // --- Build proof order (postorder) only along chosen edges ---
    vector<int> order;
    vector<pair<int,int>> st2;
    st2.push_back({1, 0});
    vector<char> vis(N + 1, 0);

    while (!st2.empty()) {
        auto [v, state] = st2.back();
        st2.pop_back();

        if (state == 0) {
            if (vis[v]) continue;
            vis[v] = 1;
            st2.push_back({v, 1});
            for (int u : chosen[v]) {
                if (!vis[u]) st2.push_back({u, 0});
            }
        } else {
            order.push_back(v);
        }
    }

    cout << order.size() << "\n";
    for (int x : order) cout << x << "\n";

    return 0;
}
```

### Izohlar

* `nth_element` ishlatdik: `need` ta eng kichik costni tez ajratadi (to‘liq sort shart emas).
* `chosen[v]` — aynan qaysi dependencylar isbotlanishini saqlaydi.
* Oxirida faqat `chosen` bo‘yicha postorder yurib, **isbotlash ketma-ketligi**ni chiqaramiz.

---

Agar xohlasangiz, men sizga shu kodni:

* `sort` bilan yanada “ko‘rinadigan” (o‘qituvchilik uchun) variantga,
* yoki grafni chizib, 1–2 ta misolni qo‘lda yurib ko‘rsatadigan variantga
  ham tayyorlab beraman.


Bu juda qiziqarli va klassik algoritmik masala bo'lib, u **Daraxtlar ustida Dinamik Dasturlash (Tree DP)** hamda **Chuqurlik bo'ylab qidirish (DFS)** usullari orqali yechiladi.

Keling, masalani qanday hal qilishni qadam-baqadam tahlil qilamiz.

### Masalaning tahlili va Algoritm

Masalada berilgan eng muhim shartlardan biri shuki: **"Kiruvchi ma'lumotlarda hech qaysi ikkita $A_{i,j}$ soni bir xil emas"**.
Bu shuni anglatadiki, har bir teorema ko'pi bilan faqat bitta boshqa teorema uchun isbot sifatida kerak bo'ladi. Agar biz teoremalar o'rtasidagi bog'liqlikni graf sifatida qursak (bu yerda qirra $A$ teoremadan $B$ teoremaga yo'nalgan, ya'ni $A$ ni isbotlash $B$ ga bog'liq), bu graf **daraxt (yoki daraxtlar o'rmoni)** ko'rinishida bo'ladi.

Asosiy maqsadimiz — **1-teoremani** isbotlash. Buning uchun quyidagi algoritmni qo'llaymiz:

1. **Dinamik dasturlash (DP) holati:** `dp[u]` — $u$-teoremani isbotlash uchun eng kamida nechta teorema isbotlanishi kerakligini bildiruvchi son.
Agar teoremaga hech qanday boshqa teorema kerak bo'lmasa (barg tugun), uni isbotlash narxi `dp[u] = 1` bo'ladi.
2. **O'tish qoidasi:**
Agar $u$-teorema $C$ ta boshqa teoremalarga bog'liq bo'lsa, shartga ko'ra ularning kamida yarmini, ya'ni $\lceil C / 2 \rceil$ tasini isbotlashimiz kerak. (Buni butun sonlar arifmetikasida `(C + 1) / 2` deb yozish mumkin).
Eng kam xarajatga erishish uchun biz $u$-teoremaning barcha farzandlari (bog'liq bo'lgan teoremalari) uchun ularning `dp` qiymatlarini hisoblab chiqamiz. Keyin bu qiymatlarni o'sish tartibida saralaymiz va eng arzon bo'lgan birinchi $\lceil C / 2 \rceil$ tasini tanlaymiz.
$dp[u] = 1 + \sum (\text{tanlangan eng arzon farzandlar})$.
3. **Ketma-ketlikni tiklash:**
Bizga faqat minimal son emas, balki qaysi tartibda isbotlash kerakligi ham so'ralgan. Buning uchun DFS yordamida har bir tugun uchun qaysi "farzand" teoremalarni tanlaganimizni xotirada saqlab boramiz. Va oxirida **post-order** (oldin farzandlarini, keyin o'zini) tartibida javobni chiqaramiz.

---

### C++ dagi yechim

Dastur 16 MB xotira limitiga va 1 soniya vaqt limitiga to'liq javob beradi. Kod tushunishga oson bo'lishi uchun izohlar bilan yozilgan.

```cpp
#include <iostream>
#include <vector>
#include <algorithm>

using namespace std;

const int MAXN = 10005;

// Har bir teorema qaysi teoremalarga bog'liqligini saqlovchi ro'yxat
vector<int> adj[MAXN]; 

// Har bir teorema uchun tanlangan eng optimal "farzand" teoremalar
vector<int> chosen[MAXN]; 

// dp[i] - i-teoremani isbotlash uchun jami qancha teorema isbotlash kerakligi
int dp[MAXN]; 

// Chuqurlik bo'ylab qidirish (DFS) va Dinamik Dasturlash
void dfs(int u) {
    // Agar teorema hech narsaga bog'liq bo'lmasa
    if (adj[u].empty()) {
        dp[u] = 1;
        return;
    }

    // Farzandlarning dp qiymatlarini va ularning raqamlarini saqlash
    vector<pair<int, int>> children_dp;
    for (int v : adj[u]) {
        dfs(v); // Oldin farzandni hisoblaymiz
        children_dp.push_back({dp[v], v});
    }

    // Eng kam teorema talab qiladiganlarini tanlash uchun saralaymiz
    sort(children_dp.begin(), children_dp.end());

    // Qancha teorema kerak? (Kamida yarmi: ceil(C/2))
    int req = (adj[u].size() + 1) / 2; 
    
    dp[u] = 1; // O'zini isbotlash uchun 1 qadam
    for (int i = 0; i < req; ++i) {
        dp[u] += children_dp[i].first;           // Narxni qo'shamiz
        chosen[u].push_back(children_dp[i].second); // Qaysi birini tanlaganimizni saqlaymiz
    }
}

// Javobni chop etish uchun (Post-order traversal)
// Teoremani isbotlashdan oldin, u bog'liq bo'lgan teoremalarni chiqarish kerak
void print_order(int u) {
    for (int v : chosen[u]) {
        print_order(v);
    }
    cout << u << "\n";
}

int main() {
    // Kiritish va chiqarishni tezlashtirish
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    int N;
    if (!(cin >> N)) return 0;

    // Kirituvchi ma'lumotlarni o'qish
    for (int i = 1; i <= N; ++i) {
        int dep;
        while (cin >> dep && dep != 0) {
            adj[i].push_back(dep);
        }
    }

    // 1-teorema (Asosiy teorema) uchun DFS ni boshlaymiz
    dfs(1);

    // Minimal teoremalar sonini chiqaramiz
    cout << dp[1] << "\n";
    
    // Qaysi tartibda isbotlash kerakligini chiqaramiz
    print_order(1);

    return 0;
}

```

### Yechimning afzalliklari:

* **Vaqt murakkabligi (Time Complexity):** $O(N \log N)$. Har bir tugunda uning farzandlarini saralash (`sort`) talab qilinadi. Barcha tugunlardagi farzandlar soni yig'indisi ko'pi bilan $N$ ga teng bo'lganligi sababli, umumiy saralash vaqti optimal bo'ladi. 10,000 ta teorema uchun bu bir necha millisoniyada ishlaydi.
* **Xotira murakkabligi (Space Complexity):** $O(N)$. `vector` va massivlar faqat $N$ element uchun ochiladi. Rekursiya (DFS) chuqurligi ham ko'pi bilan $N$ bo'lishi mumkin. Bu 16 MB xotira limiti doirasiga (taxminan 1-2 MB joy oladi) bemalol sig'adi.

Kodni tushunishda biror qism mavhum bo'lsa yoki C++ bo'yicha boshqa savollaringiz bo'lsa, bemalol so'rashingiz mumkin. Yana biron masalaga yordam kerakmi?